In [98]:
! pip install openpyxl

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [4]:
!conda install nltk -y

Channels:
 - conda-forge
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /home/ec2-user/anaconda3/envs/JupyterSystemEnv

  added / updated specs:
    - nltk


The following NEW packages will be INSTALLED:

  click              conda-forge/noarch::click-8.3.1-pyh8f84b5b_1 
  joblib             conda-forge/noarch::joblib-1.5.3-pyhd8ed1ab_0 
  nltk               conda-forge/noarch::nltk-3.9.3-pyhcf101f3_0 
  regex              conda-forge/linux-64::regex-2026.2.28-py310h7c4b9e2_0 
  tqdm               conda-forge/noarch::tqdm-4.67.3-pyh8f84b5b_0 

The following packages will be UPDATED:

  ca-certificates                       2026.1.4-hbd8a1cb_0 --> 2026.2.25-hbd8a1cb_0 
  certifi                             2026.1.4-pyhd8ed1ab_0 --> 2026.2.25-pyhd8ed1ab_0 




Preparing transaction: done
Verifying transaction: done
Executing transaction: done


In [1]:
import pandas as pd
import numpy as np
import tqdm
from typing import *

import numpy as np
import pandas as pd
from IPython.display import *
#from call_bedrock_fast_claude_v3_py import *
#import sagemaker
#import boto3
#from sagemaker import get_execution_role

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import KFold
#import sagemaker
import boto3
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler,MinMaxScaler
import matplotlib.pyplot as plt
#import torch
from transformers import BertTokenizer, BertModel
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer
import faiss
#from titan_embedding_first import *
import awswrangler as wr
#from titan_parallel import *
#from haiku_parallel import *

/home/ec2-user/anaconda3/envs/gpu_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
import awswrangler as wr

ModuleNotFoundError: No module named 'awswrangler'

In [6]:
import pandas as pd

input1=wr.s3.read_parquet('s3://idq-cradle-output-bucket/dup_output/')
input1



,asin,ptd,attribute_name,attribute_value
0,B0CKMV7YNW,ABIS_BOOK,minimum_compatible_size,
1,B0D89KN9XT,ITEM_CONTAINER,maximum_lifting_height,
2,B0DCVWMBQJ,HARDWARE_CLAMP_VISE,produce_classification_variety,
3,B0DC46PKC1,BODY_POSITIONER,t2_model_year,
4,B0C7FXF4BL,LIGHT_FIXTURE,leg_diameter,
...,...,...,...,...
26411457,B0CL3YP6C2,INPUT_MOUSE,video_output_interface,
26411458,B0DRSHLQHJ,BOTTLE,maximum_lifting_height,
26411459,B0DHSVDTC1,SCREEN_PROTECTOR,product_height_dimension,
26411460,B09Z8BPSV3,ABIS_BOOK,t3_computer_cpu_speed,


In [7]:
input1.shape

(26411462, 4)

In [10]:
input_fil=input1[input1['attribute_name'].isin([ 'aspect_ratio','binding','brand_name','color_name','contributor','dvd_region','edition','format','language_published','language_value',
    'model_number','number_of_discs','number_of_items','package_type_name','part_number','program_member','publication_date','runtime','size_name'])]
input_fil.reset_index(drop=True,inplace=True)
input_fil

,asin,ptd,attribute_name,attribute_value
0,B0D4V5G5J9,EARRING,binding,
1,9359645869,ABIS_BOOK,language_value,
2,B0CR15VWF8,ABIS_BOOK,binding,
3,B0BMYRN531,SHOES,dvd_region,
4,B0F43JFTCY,GLITTER,format,
...,...,...,...,...
527053,B0DSG9JTGD,CONDOM,binding,
527054,B0CRL9KPM7,VEHICLE_SCAN_TOOL,model_number,
527055,B0CYLTT27Z,CELLULAR_PHONE_CASE,number_of_items,
527056,B0DBPH84JD,DRESS,model_number,


In [25]:
input_fil[input_fil['attribute_value'].isna()]

,asin,ptd,attribute_name,attribute_value


In [22]:
# new_df=pd.pivot_table(input_fil,index=['asin','ptd',],columns='attribute_name',values='attribute_value',aggfunc='first').reset_index()
# new_df.columns.name = None
new_df[new_df['language_value'].notna()]

,asin,ptd,binding,dvd_region,format,language_value,model_number,number_of_items,part_number,program_member,publication_date
0,000024578X,ABIS_BOOK,,,,,,,,,
1,0000545783,ABIS_BOOK,,,,,,,,,
2,0002458942,ABIS_BOOK,,,,,,,,,
3,000713746X,ABIS_BOOK,,,,,,,,,
4,000744785X,ABIS_BOOK,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...
58540,B0F84FFFJN,SCULPTING_MATERIAL,,,,,,,,,
58541,B0F84HHBX5,ABIS_BOOK,,,,,,,,,
58542,B0F84HXMCC,ART_CRAFT_KIT,,,,,,,,,
58543,B0F85JLLYX,BED,,,,,,,,,


In [42]:
dup_it=pd.read_excel('dff0f198-9c10-4538-8842-f120548bc824.xlsx')
dup_it

,asin,gl_product_group_desc,item_name
0,B0DHRP9S7C,gl_book,Demon Slayer Vol. 8 Manga
1,B0F83Z6NF1,gl_lawn_and_garden,ibains Big Size Live Wonderful All Season Purp...
2,B0DGQB6S1V,gl_book,Tommoro Astrophysics For People In A Hurry Pop...
3,B0BZQ95Q7F,gl_sports,Bodyfit Fitness New Multi Adjustable Weight Be...
4,B0C7BWLYJ1,gl_lawn_and_garden,UR LITTLE SHOP UV Resistant 5 X 10 Feet PVC Ga...
...,...,...,...
74187,B08LCMQ6F8,gl_home,Home Tex Life Kora Grass Cushion Sleep Healthy...
74188,B0CD7CKQJM,gl_watch,ACM Watch Strap Slide 42MM 44MM 45MM 46MM 49MM...
74189,B0DFMZ48XW,gl_wireless_accessory,Amazplus Non Slip Car Mobile Holder Mat for Da...
74190,B0D8Q8LV5Y,gl_shoes,FAUSTO FST KI-664 GREY-38 Women's Grey Abstrac...


In [52]:
#master_df=pd.merge(dup_it,new_df,on='asin',how='left')
for col in master_df.columns:
    print(f'fillrate_{col}:{(len(master_df[(master_df[col]!='') & (master_df[col].notna())])/len(master_df)):.2f}%')

fillrate_asin:1.00%
fillrate_gl_product_group_desc:1.00%
fillrate_item_name:1.00%
fillrate_ptd:0.77%
fillrate_binding:0.00%
fillrate_dvd_region:0.00%
fillrate_format:0.00%
fillrate_language_value:0.00%
fillrate_model_number:0.01%
fillrate_number_of_items:0.01%
fillrate_part_number:0.01%
fillrate_program_member:0.00%
fillrate_publication_date:0.00%


In [79]:
import numpy as np
master_df['ptd'].replace([pd.NA,np.nan,'<NA>'],'unknown',inplace=True)
master_df2=master_df.replace('',pd.NA,regex=True).set_index(['asin',]).stack().reset_index(name='value')

master_df2

/tmp/ipykernel_17939/187953696.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  master_df['ptd'].replace([pd.NA,np.nan,'<NA>'],'unknown',inplace=True)


,asin,level_1,value
0,B0DHRP9S7C,gl_product_group_desc,gl_book
1,B0DHRP9S7C,item_name,Demon Slayer Vol. 8 Manga
2,B0DHRP9S7C,ptd,unknown
3,B0F83Z6NF1,gl_product_group_desc,gl_lawn_and_garden
4,B0F83Z6NF1,item_name,ibains Big Size Live Wonderful All Season Purp...
...,...,...,...
231331,B0D8Q8LV5Y,item_name,FAUSTO FST KI-664 GREY-38 Women's Grey Abstrac...
231332,B0D8Q8LV5Y,ptd,SANDAL
231333,B0DY69WDM6,gl_product_group_desc,gl_book
231334,B0DY69WDM6,item_name,the lean product playbook #bestsellerbook


In [92]:
master_df2['canonical_text']=master_df2['level_1']+':'+master_df2['value'].astype('str')
master_df3=master_df2.groupby(['asin'])['canonical_text'].agg(' '.join).reset_index()


In [139]:
#aud=pd.read_excel('dup_data_merg.xlsx',engine="openpyxl")
aud[aud['Target']=='B0F1DFDJ7X']
aud1=aud[['Target','Source','True_duplicate']]
aud1[aud1['Target']=='B0F1DFDJ7X']

,Target,Source,True_duplicate
32430,B0F1DFDJ7X,B0BTHT1V1L,yes
32856,B0F1DFDJ7X,B0D86TF3W3,no
36784,B0F1DFDJ7X,B0DPZKN9R9,no
36867,B0F1DFDJ7X,B09T2QVRD3,no
45066,B0F1DFDJ7X,B0F3TSMR4D,1
45149,B0F1DFDJ7X,B0DZX5QD1S,0
45316,B0F1DFDJ7X,B0DSC83N2C,0
45742,B0F1DFDJ7X,B0DB65C55T,0
48777,B0F1DFDJ7X,B0DB65C55T,no


In [3]:
aud=pd.read_excel('dup_data_merg.xlsx',engine="openpyxl")
aud1=aud[['Target','Source','True_duplicate']]
display(aud[aud['True_duplicate'].isna()])
aud1['True_duplicate']=(aud1['True_duplicate'].astype(str).str.strip().str.lower().replace({'yes': 1, 'no': 0, '1': 1, '0': 0}))
display(aud1[aud1['True_duplicate'].isna()])
aud1=pd.merge(aud1,master_df3,left_on='Target',right_on='asin').drop(columns='asin').rename(columns={'canonical_text':'canonical_text_tar','True_duplicate':'label'})
aud1=pd.merge(aud1,master_df3,left_on='Source',right_on='asin').drop(columns='asin').rename(columns={'canonical_text':'canonical_text_sou'})
aud1=aud1.drop_duplicates().reset_index(drop=True)

aud1=aud1.drop_duplicates().reset_index(drop=True)
aud1

,Target,Source,Beagle score,Search_impression,True_duplicate,Merge,Type,PTD,PL,GL,Assosiate,Week,Month,Task,Audited,True_duplicates,Merges,Retail Merge,3P Merge,PL Final


,Target,Source,True_duplicate


NameError: name 'master_df3' is not defined

In [7]:
import pandas as pd
aud=pd.read_excel('dup_data_merg.xlsx',engine="openpyxl")
aud

,Target,Source,Beagle score,Search_impression,True_duplicate,Merge,Type,PTD,PL,GL,Assosiate,Week,Month,Task,Audited,True_duplicates,Merges,Retail Merge,3P Merge,PL Final
0,B0DBQHPWNY,B0CP66HSKW,0.631053,3011.0,no,no,3p,SHOWERHEAD,gl_home_improvement,OHL,Kavana,1,Jan,BAU,1,NaN,NaN,NaN,NaN,OHL
1,B0BR3Z4HMX,B0BR3ZGMZ1,0.597015,3007.0,no,no,3p,OFFICE_PRODUCTS,gl_home_improvement,OHL,Kavana,1,Jan,BAU,1,NaN,NaN,NaN,NaN,OHL
2,B0CG4R3JD2,B0D3HLW97Z,0.641449,2994.0,no,no,3p,WALLPAPER,gl_home_improvement,OHL,Kavana,1,Jan,BAU,1,NaN,NaN,NaN,NaN,OHL
3,B07SZ2CFC8,B0851B8NWF,0.486046,2943.0,no,no,3p,ALARM,gl_home_improvement,OHL,Kavana,1,Jan,BAU,1,NaN,NaN,NaN,NaN,OHL
4,B0CP1435R8,B0CLYL88CZ,0.621914,2928.0,no,no,3p,BONDING_ADHESIVES,gl_biss,OHL,Kavana,1,Jan,BAU,1,NaN,NaN,NaN,NaN,OHL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84749,B0F4CGZPXK,B0DV3PBVKL,0.882581,2.0,no,no,3p,BRACELET,gl_jewelry,SL,Kavana,36,Sep,BAU,1,NaN,NaN,NaN,NaN,SL
84750,B0F4D26HHF,B0DV3QGWWT,0.881791,1.0,no,no,3p,BRACELET,gl_jewelry,SL,Kavana,36,Sep,BAU,1,NaN,NaN,NaN,NaN,SL
84751,B08CKJ571D,B08BYYM7KH,0.874797,1.0,yes,no,3p,BRACELET,gl_jewelry,SL,Kavana,36,Sep,BAU,1,1.0,NaN,NaN,NaN,SL
84752,B0DRVGHHPH,B0DRVCJ9WZ,0.870947,4.0,yes,yes,3p,EARRING,gl_jewelry,SL,Kavana,36,Sep,BAU,1,1.0,1.0,NaN,1.0,SL


In [8]:
aud1=pd.read_parquet('trainset.parquet')
aud2=pd.merge(aud1,aud[['Target','Source','True_duplicate']],on=['Target','Source'],how='left')
aud2

,Target,Source,label,canonical_text_tar,canonical_text_sou,True_duplicate
0,B0DBQHPWNY,B0CP66HSKW,no,gl_product_group_desc:gl_home_improvement item...,gl_product_group_desc:gl_home_improvement item...,no
1,B0DBQHPWNY,B0CP66HSKW,no,gl_product_group_desc:gl_home_improvement item...,gl_product_group_desc:gl_home_improvement item...,no
2,B0BR3Z4HMX,B0BR3ZGMZ1,no,gl_product_group_desc:gl_home_improvement item...,gl_product_group_desc:gl_home_improvement item...,no
3,B0BR3Z4HMX,B0BR3ZGMZ1,no,gl_product_group_desc:gl_home_improvement item...,gl_product_group_desc:gl_home_improvement item...,no
4,B0BR3Z4HMX,B0BR3ZGMZ1,no,gl_product_group_desc:gl_home_improvement item...,gl_product_group_desc:gl_home_improvement item...,no
...,...,...,...,...,...,...
103421,B0F1DLCGQM,B0F2SX1M7R,no,gl_product_group_desc:gl_jewelry item_name:Aai...,gl_product_group_desc:gl_jewelry item_name:Aai...,no
103422,B0F4CGZPXK,B0DV3PBVKL,no,gl_product_group_desc:gl_jewelry item_name:Div...,gl_product_group_desc:gl_jewelry item_name:Div...,no
103423,B0F4D26HHF,B0DV3QGWWT,no,gl_product_group_desc:gl_jewelry item_name:Div...,gl_product_group_desc:gl_jewelry item_name:Div...,no
103424,B08CKJ571D,B08BYYM7KH,yes,gl_product_group_desc:gl_jewelry item_name:Urv...,gl_product_group_desc:gl_jewelry item_name:Urv...,yes


In [72]:
import numpy as np
aud2['label']=np.where(aud2['label'].isnull(),aud2['True_duplicate'],aud2['label'])
aud2['label']=aud2['label'].replace(['Yes','No','yes','no'],[1,0,1,0],regex=True)
aud2=aud2[aud2['label']!='R']

/tmp/ipykernel_28786/3714747074.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  aud2['label']=np.where(aud2['label'].isnull(),aud2['True_duplicate'],aud2['label'])
/tmp/ipykernel_28786/3714747074.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  aud2['label']=aud2['label'].replace(['Yes','No','yes','no'],[1,0,1,0],regex=True)
/tmp/ipykernel_28786/3714747074.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See

In [73]:
aud2.drop_duplicates(inplace=True)
aud2['label'].value_counts()

label
0    45171
1    21965
Name: count, dtype: int64

In [91]:
from sklearn.model_selection import train_test_split
train_df,test_df=train_test_split(aud2,test_size=0.2, random_state=42, stratify=aud2['label'])

In [61]:
# pos_df = train_df[train_df["label"] == 1]
# neg_df = train_df[train_df["label"] == 0]
# neg_df_bal = neg_df.sample(len(pos_df), random_state=42)

# # balanced dataset
# train_df_bal = pd.concat([pos_df, neg_df_bal]).sample(frac=1, random_state=42)


In [65]:
train_df_bal

,Target,Source,label,canonical_text_tar,canonical_text_sou,True_duplicate
94260,B0CKF2SSFL,B08CR75L2R,1,gl_product_group_desc:gl_grocery item_name:Swe...,gl_product_group_desc:gl_grocery item_name:Swe...,No
1252,B0DDHB6QCM,B0DDH8JSQ5,0,gl_product_group_desc:gl_home item_name:Modern...,gl_product_group_desc:gl_home item_name:Modern...,no
95442,B0B5X4D8PK,B0BB7SSTQ7,1,gl_product_group_desc:gl_kitchen item_name:VIS...,gl_product_group_desc:gl_kitchen item_name:VIS...,yes
99450,B0DRYTCY5Y,B0DT4VGV3K,1,gl_product_group_desc:gl_home item_name:ROTTO ...,gl_product_group_desc:gl_home item_name:ROTTO ...,yes
55873,B0C8JH3Y9S,B0C7H25RTG,0,gl_product_group_desc:gl_apparel item_name:adi...,gl_product_group_desc:gl_apparel item_name:100...,no
...,...,...,...,...,...,...
23125,B0C1SJ6R4P,B0C1ZFXR7J,0,gl_product_group_desc:gl_jewelry item_name:Nat...,gl_product_group_desc:gl_jewelry item_name:Nat...,no
25476,184916276X,B0072LZA0A,0,gl_product_group_desc:gl_book item_name:SIX GR...,gl_product_group_desc:gl_book item_name:SIX GR...,No
11764,B0DL5B68PB,B0DKT384D1,0,gl_product_group_desc:gl_wireless_accessory it...,gl_product_group_desc:gl_wireless_accessory it...,No
52354,B0F2ML9S1T,B0DYVJKM3M,0,gl_product_group_desc:gl_book item_name:Icebre...,gl_product_group_desc:gl_book item_name:ICEBRE...,no


In [75]:
from sentence_transformers import SentenceTransformer,InputExample,losses
from torch.utils.data import DataLoader
from datasets import Dataset


train_examp=[InputExample(texts=[row.canonical_text_tar,row.canonical_text_sou],label=float(row.label)) for row in train_df.itertuples()]


In [2]:
import sentence_transformers
import transformers
import huggingface_hub

print(sentence_transformers.__version__)
print(transformers.__version__)
print(huggingface_hub.__version__)

5.2.0
4.44.2
0.36.2


In [3]:
# from sentence_transformers import SentenceTransformer, InputExample
# from sentence_transformers.losses import ContrastiveLoss
# from torch.utils.data import DataLoader

# # Prepare training examples
# train_examples = [
#     InputExample(texts=[s1, s2], label=float(l))
#     for s1, s2, l in zip(
#         train_df["canonical_text_tar"],
#         train_df["canonical_text_sou"],
#         train_df["label"]
#     )
# ]

# # batch size
# train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=32)

# # Load model
# model = SentenceTransformer("all-mpnet-base-v2")

# # Define loss
# train_loss = ContrastiveLoss(model)

# # warmup calculation (based on ratio)
# warmup_steps = int(len(train_dataloader) * 3 * 0.1)

# # Train
# model.fit(
#     train_objectives=[(train_dataloader, train_loss)],
#     epochs=3,
#     warmup_steps=warmup_steps,
#     optimizer_params={"lr": 2e-5}
# )

# # Save model
# model.save("fine_tuned_mpnet")

In [79]:

from sentence_transformers.losses import ContrastiveLoss
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from datasets import Dataset

# Prepare dataset
data = {
    "sentence1": train_df["canonical_text_tar"].tolist(),
    "sentence2": train_df["canonical_text_sou"].tolist(),
    "label": train_df["label"].astype(float).tolist()
}

train_dataset = Dataset.from_dict(data)

# Load model
model = SentenceTransformer("all-mpnet-base-v2")

# Define loss
train_loss = ContrastiveLoss(model,margin=0.9)

# Training arguments
training_args = SentenceTransformerTrainingArguments(
    output_dir="embedding_model",
    num_train_epochs=4,
    per_device_train_batch_size=32,
    warmup_ratio=0.1,
    learning_rate=2e-5,
    logging_steps=100,
    eval_strategy="no"
)
# Trainer
trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    loss=train_loss
)

# Train
trainer.train()

# Save model
model.save("fine_tuned_mpnet")

/home/ec2-user/anaconda3/envs/gpu_env/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
                                                                     

Step,Training Loss
100,0.121700
200,0.077100
300,0.071300
400,0.070400
500,0.068700
600,0.066500
700,0.067600
800,0.068200
900,0.068100
1000,0.065500


In [94]:
test_df1=test_df#.sample(1000)
test_df1.reset_index(drop=True,inplace=True)
test_df1


,Target,Source,label,canonical_text_tar,canonical_text_sou,True_duplicate
0,B0DHRZ1HDC,B0CZTT4SVK,0,gl_product_group_desc:gl_book item_name:Glucos...,gl_product_group_desc:gl_biss item_name:RAVIZA...,no
1,B0F5JKHXV7,B0F5J9BH39,1,gl_product_group_desc:gl_luggage item_name:Bus...,gl_product_group_desc:gl_luggage item_name:Bus...,yes
2,B0DPJ167L3,B0DPHZ47ND,1,gl_product_group_desc:gl_kitchen item_name:B S...,gl_product_group_desc:gl_kitchen item_name:B S...,yes
3,B0F1DBN88M,B0CDC9CJR2,0,gl_product_group_desc:gl_book item_name:PRITIS...,gl_product_group_desc:gl_home item_name:LIODOR...,no
4,B0DDCNBNGD,B0DVHV1Y3Q,0,gl_product_group_desc:gl_book item_name:NEWYE ...,gl_product_group_desc:gl_home item_name:Automa...,no
...,...,...,...,...,...,...
13423,B09W36CL5P,B09W3XQTGB,0,gl_product_group_desc:gl_wireless_accessory it...,gl_product_group_desc:gl_wireless_accessory it...,No
13424,B0DXVFWC44,B0C5JWRSDH,1,gl_product_group_desc:gl_baby_product item_nam...,gl_product_group_desc:gl_baby_product item_nam...,Yes
13425,B00AQ7G70O,B09VKY7CVP,0,gl_product_group_desc:gl_home item_name:Nakoma...,gl_product_group_desc:gl_office_product item_n...,no
13426,B0DHL5KN2T,B09MWB1DJT,1,gl_product_group_desc:gl_lawn_and_garden item_...,gl_product_group_desc:gl_lawn_and_garden item_...,yes


In [95]:
import numpy as np
from sentence_transformers import util
def evaluate_sim(model,df):
    tar=df['canonical_text_tar'].tolist()
    sor=df['canonical_text_sou'].tolist()

    tar_emb=model.encode(tar,batch_size=64,normalize_embeddings=True,convert_to_numpy=True)
    sour_emb=model.encode(sor,batch_size=64,normalize_embeddings=True,convert_to_numpy=True)
    sim=util.cos_sim(tar_emb,sour_emb).diagonal()
    return sim.tolist()

test_df1['sim']= evaluate_sim(model,test_df1)

test_df1    
    

,Target,Source,label,canonical_text_tar,canonical_text_sou,True_duplicate,sim
0,B0DHRZ1HDC,B0CZTT4SVK,0,gl_product_group_desc:gl_book item_name:Glucos...,gl_product_group_desc:gl_biss item_name:RAVIZA...,no,0.237507
1,B0F5JKHXV7,B0F5J9BH39,1,gl_product_group_desc:gl_luggage item_name:Bus...,gl_product_group_desc:gl_luggage item_name:Bus...,yes,0.733231
2,B0DPJ167L3,B0DPHZ47ND,1,gl_product_group_desc:gl_kitchen item_name:B S...,gl_product_group_desc:gl_kitchen item_name:B S...,yes,0.598332
3,B0F1DBN88M,B0CDC9CJR2,0,gl_product_group_desc:gl_book item_name:PRITIS...,gl_product_group_desc:gl_home item_name:LIODOR...,no,0.023306
4,B0DDCNBNGD,B0DVHV1Y3Q,0,gl_product_group_desc:gl_book item_name:NEWYE ...,gl_product_group_desc:gl_home item_name:Automa...,no,0.242163
...,...,...,...,...,...,...,...
13423,B09W36CL5P,B09W3XQTGB,0,gl_product_group_desc:gl_wireless_accessory it...,gl_product_group_desc:gl_wireless_accessory it...,No,-0.081873
13424,B0DXVFWC44,B0C5JWRSDH,1,gl_product_group_desc:gl_baby_product item_nam...,gl_product_group_desc:gl_baby_product item_nam...,Yes,0.658146
13425,B00AQ7G70O,B09VKY7CVP,0,gl_product_group_desc:gl_home item_name:Nakoma...,gl_product_group_desc:gl_office_product item_n...,no,0.353919
13426,B0DHL5KN2T,B09MWB1DJT,1,gl_product_group_desc:gl_lawn_and_garden item_...,gl_product_group_desc:gl_lawn_and_garden item_...,yes,0.498587


In [99]:
test_df1['pred']=(test_df1['sim']>0.75).astype(int)

In [100]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,confusion_matrix
y_true=test_df1['label'].astype(int)
y_pred=test_df1['pred']
accuracy=accuracy_score(y_true,y_pred)
precision=precision_score(y_true,y_pred)
recall=recall_score(y_true,y_pred)
cm=confusion_matrix(y_true,y_pred)

print(f'accuracy:{accuracy}')
print(f'precision:{precision}')
print(f'recall:{recall}')
print(f'cm:{cm}')

accuracy:0.6301012809055705
precision:0.4540653008962868
recall:0.6458001365809242
cm:[[5624 3411]
 [1556 2837]]


In [ ]:
# model=SentenceTransformer('all-mpnet-base-v2')
# train_dataload=DataLoader(train_examp,shuffle=True,batch_size=32)
# train_loss=losses.ContrastiveLoss(model)
# model.fit(train_objectives=[(tra`in_dataload,train_loss)],epochs=1,warmup_steps=int(len(train_dataload) * 0.1),
#     show_progress_bar=True)

In [11]:
!nvidia-smi

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Fri Mar  6 13:44:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10G                    On  |   00000000:00:1E.0 Off |                    0 |
|  0%   23C    P8             15W /  300W |       0MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.5.1.post107
None
False


In [2]:
df=pd.read_csv('s3://biswa--test/Unsaved/2026/01/22/acf66075-ea25-4318-917d-a67bf74ad35a.csv')
df.head(10)

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:298: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


,asin,rule_applied,attribute_value,ptd,status
0,B0G4HCJ732,title,mini harmonium 32 keys bass male 2 line 5 kg w...,MUSICAL_INSTRUMENTS,mandatory_
1,B0G4HCJ732,title,mini harmonium 32 keys bass male 2 line 5 kg w...,MUSICAL_INSTRUMENTS,mandatory_
2,B0G4HCJ732,title,mini harmonium 32 keys bass male 2 line 5 kg w...,MUSICAL_INSTRUMENTS,mandatory_
3,B0G4HCJ732,title,mini harmonium 32 keys bass male 2 line 5 kg w...,MUSICAL_INSTRUMENTS,pace_av
4,B0G4HCJ732,title,mini harmonium 32 keys bass male 2 line 5 kg w...,MUSICAL_INSTRUMENTS,relevant25_sf_band_c_rac_tech_spec_po_PACE_REF_25
5,B0G4HCJ732,title,mini harmonium 32 keys bass male 2 line 5 kg w...,MUSICAL_INSTRUMENTS,pace_av
6,B0G4HCJ732,title,mini harmonium 32 keys bass male 2 line 5 kg w...,MUSICAL_INSTRUMENTS,rac_tech_spec_po_
7,B0G4HCJ732,title,mini harmonium 32 keys bass male 2 line 5 kg w...,MUSICAL_INSTRUMENTS,mandatory_
8,B0G4HCJ732,title,mini harmonium 32 keys bass male 2 line 5 kg w...,MUSICAL_INSTRUMENTS,mandatory_
9,B0G4HCJ732,title,mini harmonium 32 keys bass male 2 line 5 kg w...,MUSICAL_INSTRUMENTS,mandatory_


In [3]:
df.drop_duplicates(inplace=True)

In [22]:
df[df['rule_applied']=='size']
df['rule_applied'].unique()

array(['title', 'polar_pattern', 'compatible_devices', 'color_name',
       'connector_type', 'connectivity_technology', 'power_source_type',
       'special_features', 'body_material_type', 'material_type',
       'finish_type', 'neck_material_type', 'number_of_strings',
       'string_material_type', 'back_material_type',
       'guitar_bridge_system', 'hand_orientation', 'top_material_type',
       'fretboard_material_type', 'guitar_pickup_configuration',
       'scale_length', 'style_name', 'number_of_keys',
       'age_range_description', 'prod_desc', 'noise_level',
       'number_of_channels'], dtype=object)

In [32]:
df1.columns
df.sort_values('asin',inplace=True)

In [4]:
df1=df.head(600)
df2=df.pivot_table(index=['asin','ptd'],columns='rule_applied',values='attribute_value',aggfunc='first').reset_index()


In [5]:
print(df2.shape)
len(df2.columns)


(227343, 35)


35

In [103]:
embed_df[embed_df['asin']=='B0002H0KGK']

rule_applied,asin,ptd,age_range_description,back_material_type,body_material_type,brand_name,color_name,compatible_devices,connectivity_technology,connector_type,...,power_source_type,prod_desc,scale_length,special_features,string_material_type,style_name,title,top_material_type,comb_columns,len
2639,B0002H0KGK,GUITARS,,rosewood,spruce,yamaha,natural,,,,...,,,,,stainless steel,,"yamaha f310p 41"" acoustic guitar package, natu...",spruce wood,rosewood spruce yamaha natural rosewood right ...,211


In [6]:
#df2.drop(columns='comb_columns',inplace=True)
df2['comb_columns']=''
for i in range(2,len(df2.columns)-1):
    print(df2.columns[i])
    df2[df2.columns[i]]= df2[df2.columns[i]].fillna(' ')
    if i<len(df2.columns)-1:
        df2['comb_columns']=df2['comb_columns']+df2[df2.columns[i]]+' '

age_range_description
back_material_type
body_material_type
brand_name
color_name
compatible_devices
connectivity_technology
connector_type
finish_type
fretboard_material_type
guitar_bridge_system
guitar_pickup_configuration
hand_orientation
hardware_interface
instrument
instrument_key
material_type
model_name
model_year
neck_material_type
noise_level
number_of_channels
number_of_keys
number_of_strings
polar_pattern
power_source_type
prod_desc
scale_length
special_features
string_material_type
style_name
title
top_material_type


In [7]:
import re
#df2['comb_columns'].replace('        '," ",inplace=True,regex=True)
df2['comb_columns']=df2['comb_columns'].apply(lambda x: re.sub(r"\s+", " ",x).strip())
df2['comb_columns'][0].strip()


'wood yamaha black wood customer relationship marketing theoretical and managerial perspectives'

In [73]:
completeness = (
    df2.notnull().mean()
    .sort_values(ascending=False)
    .reset_index(name="non_null_pct")
)
# print(completeness)
# df2.notnull().mean().sort_values(ascending=False)


In [16]:
! pip install faiss-cpu

  Using cached faiss_cpu-1.12.0.tar.gz (69 kB)
  Installing build dependencies ... error
  error: subprocess-exited-with-error
  
  × installing build dependencies for faiss-cpu did not run successfully.
  │ exit code: 1
  ╰─> [48 lines of output]
        Using cached setuptools-80.10.2-py3-none-any.whl.metadata (6.6 kB)
        Using cached wheel-0.46.3-py3-none-any.whl.metadata (2.4 kB)
        Using cached numpy-2.4.2.tar.gz (20.7 MB)
        Installing build dependencies: started
        Installing build dependencies: finished with status 'done'
        Getting requirements to build wheel: started
        Getting requirements to build wheel: finished with status 'done'
        Installing backend dependencies: started
        Installing backend dependencies: finished with status 'done'
        Preparing metadata (pyproject.toml): started
        Preparing metadata (pyproject.toml): finished with status 'error'
        error: subprocess-exited-with-error
      
        × Preparing me

In [6]:
pip install sentence_transformers

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.2.1.3-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.5.147-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.6.1.9-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.3.1.170-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB

In [2]:
import pandas as pd
embed_df=pd.read_parquet('dup_raw.parquet')
#embed_df.to_parquet('dup_raw.parquet')

In [3]:
que=embed_df[embed_df['asin'].isin(['B000I1Q4TC','B000WITGQO','B0039V8WO8','B00CDJ8GDE','B00CFOX420','B00DCYMVB2','B00DHSK8C2','B00EJF5Y26','B00H1LG0YQ','B00I0Q8IX2',
    'B00I0Q8K4O','B00I2J4TWG','B00I2J4V5G','B00I2J4WJG','B00IBIVLKQ','B00ICMQWLY','B00ILALQ4K','B00O50JI0O','B00PUH0QIO','B00REF69TU',
    'B00REF6CII','B00RW19OXY','B011RI0P6C','B01HVX5YNQ','B075Q5C41K','B077TWMH2F','B079SY4HPZ','B07BSM7PFL','B07KJMJGRB','B07KXV29PM',
    'B07R4T3FKH','B07XCL6PWG','B07XY41YN3','B07YX722C2','B07ZVQ4ZCK','B081B5CQQP','B081B5T151','B082H2XD4B','B082WKG3BS','B082X8VKFB',
    'B083JG34K8','B084JSBPDS','B0854C7M9R','B087JT3DNM','B08863T6SV','B08Q3TT2ND','B08Y7RXBD2','B08Y7VMQ53','B093X1YG2N','B09HMV5PP4',
    'B09SQ8Z3MZ','B09SQCGT52','B09TSZNN1G','B09VTK9PLQ','B09VTKC91H','B09VTKQYBN','B09VTM9Z6S','B0BSLV2TYC','B0BSLVRRV8','B0C1VZR99C',
    'B0CKTR88Y7','B0CKTSZJCV','B0CKTTQMQ8','B0CS37C9RP'])]    

In [12]:
# que.reset_index(drop=True,inplace=True)
que.shape

(52, 37)

In [9]:
texts = que["comb_columns"].tolist()
embeddings_que = model.encode(
    texts,
    
    convert_to_numpy=True,
    normalize_embeddings=True,
    
    show_progress_bar=True
)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [8]:
# df2['len']=df2['comb_columns'].str.len()
# embed_df=df2[df2['len']>50]
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print('done')
texts = embed_df["comb_columns"].tolist()
# embeddings = model.encode(
#     texts,
#     batch_size=64,
#     show_progress_bar=True,
#     normalize_embeddings=True)

embeddings = model.encode(
    texts,
    
    convert_to_numpy=True,
    normalize_embeddings=True,
    
    show_progress_bar=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

done


Batches:   0%|          | 0/7027 [00:00<?, ?it/s]

In [ ]:
from titan_embedding_first import *
#import awswrangler as wr
from titan_parallel import *
model_id='amazon.titan-embed-text-v2:0'
embeddings=create_titan_embeddings(texts, model_id, batch_size=32, max_workers=5)

Total records: 224861
Processing with 5 workers, batch size 32...
Using model: amazon.titan-embed-text-v2:0
[Info] Using embedding size: 1024


Embedding Progress:   3%|▎         | 6162/224861 [01:32<54:29, 66.90it/s]  


In [10]:
embeddings.to_csv('dup_test.txt',sep='\t')

AttributeError: 'numpy.ndarray' object has no attribute 'to_csv'

In [18]:
import faiss
print(faiss.get_num_gpus())
import numpy as np

dim=embeddings.shape[1]
res=faiss.StandardGpuResources()
index=faiss.IndexFlatIP(dim)
#index=faiss.index_cpu_to_gpu(res,0,index_cpu)
index.add(embeddings)
print("Total vectors in index:", index.ntotal)

0


AttributeError: module 'faiss' has no attribute 'StandardGpuResources'

In [6]:
k = 50
D, I = index.search(embeddings[i:i+1], k)  # single ASIN
neighbors = sku_ids[I[0]]
scores = D[0]

NameError: name 'i' is not defined

In [19]:
import faiss
import numpy as np
from tqdm import tqdm

# =========================
# Inputs
# =========================
# embeddings: numpy array, shape=(220000, dim), float32, L2-normalized
# sku_ids: list or array of SKUs, length=220000
# You must normalize embeddings beforehand if using cosine similarity

embeddings = embeddings.astype(np.float32)
faiss.normalize_L2(embeddings)

dim = embeddings.shape[1]
k = 50               # Top-K neighbors
batch_size = 5000    # Tune based on RAM

# =========================
# Build CPU FAISS index
# =========================
index = faiss.IndexFlatIP(dim)   # exact inner product search
index.add(embeddings)
print("Total vectors in index:", index.ntotal)

# =========================
# Batch duplicate search
# =========================
duplicate_pairs = []

for start in tqdm(range(0, len(embeddings_que), batch_size)):
    end = min(start + batch_size, len(embeddings_que))
    batch_vecs = embeddings_que[start:end]        # shape (batch_size, dim)
    
    # Batch Top-K search
    D, I = index.search(batch_vecs, k)       # D: similarities, I: indices

    for i in range(end - start):
        query_sku = que["asin"].to_numpy()[start + i]
        neighbors = embed_df["asin"].to_numpy()[I[i]]             # map indices to SKU
        scores = D[i]

        # Filter self-match + similarity threshold
        duplicates = [(sku, score) for sku, score in zip(neighbors, scores)
                      if sku != query_sku and score >=0.1]

        for dup_sku, dup_score in duplicates:
            duplicate_pairs.append((query_sku, dup_sku, dup_score))

print("Total duplicate pairs found:", len(duplicate_pairs))

# =========================
# Optional: remove symmetric duplicates
# =========================
seen = set()
filtered_pairs = []
for a, b, s in duplicate_pairs:
    key = tuple(sorted([a, b]))
    if key not in seen:
        seen.add(key)
        filtered_pairs.append((a, b, s))

print("Filtered unique duplicate pairs:", len(filtered_pairs))


Total vectors in index: 224861


100%|██████████| 1/1 [00:02<00:00,  2.75s/it]

Total duplicate pairs found: 2548
Filtered unique duplicate pairs: 2433


In [21]:
dup_df=pd.DataFrame(filtered_pairs,columns=['asin1','asin2','prob'])
pairs_df=dup_df[dup_df['prob']>0.7]
pairs_df

,asin1,asin2,prob
0,B000I1Q4TC,B08JH82SYM,0.876521
1,B000I1Q4TC,B0C1HY7HCY,0.874083
2,B000I1Q4TC,B08NTNCS8F,0.866261
3,B000I1Q4TC,B00493WYB2,0.863470
4,B000I1Q4TC,B07MZF6YMG,0.862350
...,...,...,...
2428,B0CS37C9RP,B07ZTSYYX4,0.788774
2429,B0CS37C9RP,B07VLP9JZT,0.787538
2430,B0CS37C9RP,B06XT4N6WT,0.787060
2431,B0CS37C9RP,B083P7GWR8,0.786422


In [67]:
que[que['asin']=='B0002CZQS2']
embeded['combco']

rule_applied,asin,ptd,age_range_description,back_material_type,body_material_type,brand_name,color_name,compatible_devices,connectivity_technology,connector_type,...,power_source_type,prod_desc,scale_length,special_features,string_material_type,style_name,title,top_material_type,comb_columns,len


In [116]:
dup_df[dup_df['asin1']=='B0BSLV2TYC']
dup_df.sort_values('prob',ascending=False)
#dup_df[dup_df['asin1']=='B01HVX5YNQ']

,asin1,asin2,prob
677,B0BSLV2TYC,B0BSLVRRV8,0.994008
584,B09VTKQYBN,B09VTM9Z6S,0.969389
487,B09VTK9PLQ,B09VTKQYBN,0.957292
536,B09VTKC91H,B09VTM9Z6S,0.945362
631,B09VTM9Z6S,B0BSLV2TYC,0.936852
...,...,...,...
142,B0039V8WO8,B0BRFQ8TLL,0.561289
143,B0039V8WO8,B00CP3GNT2,0.561262
144,B0039V8WO8,B0CKC15B1V,0.561096
145,B0039V8WO8,B092DGQFJ3,0.560216


In [35]:
df.head(100).to_csv('sample.csv',index=False)

In [22]:
import numpy as np

# pairs_df: asin1, asin2
# master_df: asin + attribute columns

df = (pairs_df
      .merge(embed_df, left_on="asin1", right_on="asin", how="left", suffixes=("", "_1"))
      .merge(embed_df, left_on="asin2", right_on="asin", how="left", suffixes=("_1", "_2"))
)

attr_cols = [c for c in embed_df.columns if (c != "asin") & (c != "prod_desc") & (c != "title")]

mismatch_cols = []
final_status = []

for _, r in df.iterrows():
    mismatches = []
    for c in attr_cols:
        v1, v2 = r.get(f"{c}_1"), r.get(f"{c}_2")

        if pd.isna(v1) or pd.isna(v2) or str(v1).strip() == "" or str(v2).strip() == "":
            continue  # skip if either blank
        if str(v1).lower() != str(v2).lower():
            mismatches.append(c)

    mismatch_cols.append(mismatches)
    final_status.append("MISMATCH" if mismatches else "MATCH")

df["final_status"] = final_status
df["mismatched_columns"] = mismatch_cols
df["final_status"]=np.where((df["mismatched_columns"].astype('str')=="['comb_columns', 'len']") | (df["mismatched_columns"].astype('str')=="['comb_columns']") | (df["mismatched_columns"].astype('str')=="['len']"),'Partial_Match',df["final_status"])


In [91]:
#df[df["mismatched_columns"].astype('str')=="['comb_columns', 'len']"]

In [23]:
df.head(100)

,asin1,asin2,prob,asin_1,ptd_1,age_range_description_1,back_material_type_1,body_material_type_1,brand_name_1,color_name_1,...,scale_length_2,special_features_2,string_material_type_2,style_name_2,title_2,top_material_type_2,comb_columns_2,len_2,final_status,mismatched_columns
0,B000I1Q4TC,B08JH82SYM,0.876521,B000I1Q4TC,GUITARS,,mahogany wood,mahogany,yamaha,natural,...,,,nylon,,cordoba c1m-ce acoustic-electric cutaway nylon...,spruce wood,mahogany wood rosewood cordoba natural rosewoo...,170,MISMATCH,"[body_material_type, brand_name, guitar_bridge..."
1,B000I1Q4TC,B0C1HY7HCY,0.874083,B000I1Q4TC,GUITARS,,mahogany wood,mahogany,yamaha,natural,...,,,nylon,,cordoba music group fusion 12 6 string acousti...,spruce wood,mahogany wood rosewood cordoba natural rosewoo...,170,MISMATCH,"[body_material_type, brand_name, guitar_bridge..."
2,B000I1Q4TC,B08NTNCS8F,0.866261,B000I1Q4TC,GUITARS,,mahogany wood,mahogany,yamaha,natural,...,,,bronze,,cordoba fusion 5 nylon string acoustic-electri...,spruce wood,mahogany wood rosewood cordoba natural rosewoo...,156,MISMATCH,"[body_material_type, brand_name, guitar_bridge..."
3,B000I1Q4TC,B00493WYB2,0.863470,B000I1Q4TC,GUITARS,,mahogany wood,mahogany,yamaha,natural,...,,,nylon,spruce top,yamaha cg192s spruce top classical guitar,"spruce top,mahogany,spruce",rosewood wood yamaha natural ebony adjustable ...,148,MISMATCH,"[back_material_type, body_material_type, fretb..."
4,B000I1Q4TC,B07MZF6YMG,0.862350,B000I1Q4TC,GUITARS,,mahogany wood,mahogany,yamaha,natural,...,648,,nylon,standard,"cordoba guitars, c5 sp 6-string classical guit...",solid engelmann spruce,mahogany wood mahogany cordoba natural pau fer...,197,MISMATCH,"[brand_name, fretboard_material_type, guitar_b..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,B000WITGQO,B0CSLVX698,0.752444,B000WITGQO,GUITARS,,meranti wood,rosewood,yamaha,tobacco sunburst,...,,,bronze,,fa-25ce dreadnought acoustic electric guitar s...,rosewood,maple wood rosewood fender sunbrust walnut woo...,139,MISMATCH,"[back_material_type, brand_name, color_name, f..."
96,B000WITGQO,B09YHC3LX6,0.752237,B000WITGQO,GUITARS,,meranti wood,rosewood,yamaha,tobacco sunburst,...,41,,phosphor bronze,standard pack,intern int-p41c-bk 41-inch acoustic guitar for...,basswood,"basswood meranti, engineered wood, rosewood in...",345,MISMATCH,"[back_material_type, body_material_type, brand..."
97,B000WITGQO,B0FPL76LJT,0.752118,B000WITGQO,GUITARS,,meranti wood,rosewood,yamaha,tobacco sunburst,...,63.5,,stainless steel,,"yamaha f280 acoustic guitar – 6-string, black,...",spruce wood,rosewood redwood yamaha black rosewood fixed s...,274,MISMATCH,"[back_material_type, body_material_type, color..."
98,B0039V8WO8,B08233LB63,0.740776,B0039V8WO8,MUSICAL_INSTRUMENTS,,,metallic,yamaha,black,...,,,,classic,yamaha dtxm12 multi-pad – compact electronic d...,,black usb classic yamaha dtxm12 multi-pad – co...,171,MISMATCH,"[ptd, comb_columns, len]"


In [24]:
df.to_csv('yamaha_dup_attribute_level.csv',index=False)
pairs_df.to_csv('yamaha_onlypairs_level.csv',index=False)

In [123]:
que[que['asin']=='B01HVX5YNQ'].index[0]

6

In [32]:
import numpy as np

# ---- replace these ----
a = "B00I0Q8IX2"
b = "B00NXAV04O"
# -----------------------

a = que[que['asin']==a].index[0]      # replace with correct index for SKU A
b = embed_df[embed_df['asin']==b].index[0]  # replace with correct index for SKU B

# extract embeddings
emb_a = np.array(embeddings[a], dtype='float32')
emb_b = np.array(embeddings[b], dtype='float32')

# normalize (required for cosine similarity)
emb_a /= np.linalg.norm(emb_a)
emb_b /= np.linalg.norm(emb_b)

# cosine similarity
similarity_score = float(np.dot(emb_a, emb_b))

print(f"Similarity score (A vs B): {similarity_score}")


Similarity score (A vs B): 0.22977378964424133


In [127]:
#df[df['asin_1']=='B01HVX5YNQ']

In [45]:
display(embed_df[embed_df['asin'].isin(['B00NXAV04O',])]['comb_columns'])

print(embed_df.loc[16361,'comb_columns'])
print(embed_df.loc[18010,'comb_columns'])

18010    auxiliary 12 corded electric yamaha mixing con...
Name: comb_columns, dtype: object

auxiliary 12 corded electric d-pre class-a mic preamps for the best possible sound quality. 1-knob compression on channels 1 - 4. high quality spx effects. aux sends and assignable group sends on all channels. balanced xlr and 1/4" main outputs yamaha mg12xu 12 in pa mixer & usb audio interface - new, black
auxiliary 12 corded electric yamaha mixing console mg series - mg12xu
